# Embryo workflow

单个或批量 measurement CSV → RF / SVM 分类；fixed embryo TIFF + ROI → Fiji 邻居计数 → Python 分布图和文件输出。

选择 `EmbryoAnalyser/.venv/Scripts/python.exe` 作为运行环境，从项目目录或本 notebook 所在目录启动。只需更改下面的输入配置；所有输出默认写入项目根目录 `outputs/notebook`。

In [ ]:
from pathlib import Path
import json
import sys

working = Path.cwd().resolve()
ROOT = next((path for path in [working, *working.parents]
             if (path / 'EmbryoAnalyser/final_analyser.ipynb').is_file()), None)
if ROOT is None:
    raise RuntimeError('请从 summer_project 或 EmbryoAnalyser 目录启动 notebook。')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display, Image
from EmbryoAnalyser.workflow import train_workflow, classify_workflow, distribution_workflow
from EmbryoAnalyser.neighbours import run_neighbours

print('项目：', ROOT)


## 输入配置

列表中可填一个文件、多个文件，或递归目录。`MODELS` 可选 `both`、`rf`、`svm`。默认明确使用 gap43 的 `train` 目录训练。

设置 `RETRAIN=False` 可复用已保存模型。两个 `RUN_...` 开关可以分别选择分类与图片流程。图片需配套 ROI ZIP，当前不包含自动分割。

In [ ]:
TRAIN_INPUTS = [ROOT / 'dataset/raw_dataset/gap43-mCherry/train']
CSV_INPUTS = [ROOT / 'dataset/raw_dataset/E-CadGFP']
IMAGE_INPUTS = [ROOT / 'dataset/fixed_EM/processed/del15/s9_1/Composite.tif']

MODELS = 'both'
OUTPUT = ROOT / 'outputs/notebook'
MODEL_FILE = OUTPUT / 'training/models.joblib'
RETRAIN = True
RUN_CLASSIFICATION = True
RUN_FIJI = True
FIJI_TIMEOUT = None  # 默认不限时；可设为整个图片批次的秒数上限


## CSV 分类

每个 CSV 对应一个胚胎；模型沿用原清理、分箱和训练参数。输出 `0=control`、`1=mutant`。预测无需标签，只有可识别的真实标签才参与报告指标。

In [ ]:
if RUN_CLASSIFICATION:
    if RETRAIN:
        trained = train_workflow(TRAIN_INPUTS, OUTPUT / 'training', model=MODELS)
        MODEL_FILE = trained['model_file']
        print('训练胚胎：', trained['report']['training_embryos'])
        print('模型保存于：', MODEL_FILE)
    classified = classify_workflow(CSV_INPUTS, MODEL_FILE, OUTPUT / 'classification')
    display(classified['predictions'])
    print('预测 CSV：', classified['predictions_csv'])
    print('验证指标：', json.dumps(classified['report']['metrics'], ensure_ascii=False, indent=2))


## Fiji 邻居计数和 Python 分布

图片和 ROI 会复制到新的运行目录。计数在独立 Fiji 界面中执行，完成后自动退出；工作宏仅替换目录选择语句。结果包含逐细胞邻居数、频数和百分比 CSV、PNG、PDF 与批次汇总。

In [ ]:
if RUN_FIJI:
    neighbours = run_neighbours(IMAGE_INPUTS, OUTPUT / 'neighbours',
                                timeout=FIJI_TIMEOUT, show=False)
    display(pd.read_csv(neighbours['summary_csv']))
    for sample in neighbours['samples']:
        print('输入：', sample['input_image'])
        display(pd.read_csv(sample['distribution_csv']))
        display(Image(filename=sample['png']))
    print('批次汇总：', neighbours['summary_csv'])


## 可选：直接查看已有计数 CSV

已有 `slow_neighbour_counting.csv` 可直接用 Python 绘图，无需再次运行 Fiji。更改 `EXISTING_COUNTS` 为单个 CSV、多 CSV 或目录；设 `PLOT_EXISTING_COUNTS=True` 后运行。

In [ ]:
PLOT_EXISTING_COUNTS = False
EXISTING_COUNTS = [ROOT / 'dataset/fixed_EM/processed/del15/s9_1/slow_neighbour_counting.csv']

if PLOT_EXISTING_COUNTS:
    plotted = distribution_workflow(EXISTING_COUNTS, OUTPUT / 'existing_distributions')
    for sample in plotted['samples']:
        print('输入：', sample['input_csv'])
        display(pd.read_csv(sample['distribution_csv']))
        display(Image(filename=sample['png']))
    print('汇总：', plotted['summary_csv'])
